# Agent Loop Demo (No Memory) — LangChain + Azure OpenAI

**Goal of this notebook:** see the 5-step agent loop *by itself*, with nothing else mixed in.

> Perceive -> Plan -> Act -> Observe -> Reflect -> (loop back OR stop)

We are **not** using Memory in this version on purpose. This lets you clearly see:
- exactly what each of the 5 steps does on its own
- how the loop goes back to step 1 (Perceive) when the goal isn't met yet
- how the loop stops when the goal is met
- **the limitation** of not having memory (shown at the end)

**Example task:** *"If the stock price drops below Rs 100, send an alert."*

To keep the demo simple and dependency-free for class, we simulate the stock price with random numbers instead of calling a real stock API. Swap in a real price source later — the loop logic doesn't change.


## 0. Setup
Load Azure OpenAI credentials from your `.env` file.

In [1]:
# pip install langchain langchain-openai python-dotenv

import os
import time
import random
from dotenv import load_dotenv

from langchain_openai import AzureChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

load_dotenv()  # reads AZURE_OPENAI_ENDPOINT, AZURE_OPENAI_API_KEY, etc. from your .env file

llm = AzureChatOpenAI(
    azure_endpoint=os.environ["AZURE_OPENAI_ENDPOINT"],
    api_key=os.environ["AZURE_OPENAI_API_KEY"],
    azure_deployment=os.environ["AZURE_OPENAI_DEPLOYMENT"],
    api_version=os.environ.get("AZURE_OPENAI_API_VERSION", "2024-10-21"),
    temperature=0,
)

PRICE_THRESHOLD = 100.0
print("Setup done. Threshold price: Rs", PRICE_THRESHOLD)


Setup done. Threshold price: Rs 100.0


## 1. PERCEIVE
"Look at what's happening right now."

Here: check the current stock price. (Simulated for this demo.)

In [12]:
def perceive():
    """Look at what's happening right now: get the current price."""
    current_price = round(random.uniform(80, 120), 2)   # simulated price
    print(f"[PERCEIVE] Current price: Rs {current_price}")
    return current_price

# quick test
perceive()


[PERCEIVE] Current price: Rs 109.88


109.88

## 2. PLAN
"Decide what to do next."

Here: ask the LLM (via LangChain) whether the price is below the threshold, and what to do about it. This is the ONLY step that uses LangChain + Azure OpenAI — the rest are plain Python.

In [13]:
plan_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are the planning step of a simple agent. "
     "You are told the current stock price and a threshold. "
     "If the price is BELOW the threshold, reply with exactly one word: SEND_EMAIL. "
     "Otherwise, reply with exactly one word: WAIT. "
     "Do not explain. Only output that one word."),
    ("user", "Current price: Rs {price}. Threshold: Rs {threshold}."),
])

plan_chain = plan_prompt | llm | StrOutputParser()

def plan(current_price):
    """Decide what to do next, given the current price."""
    decision = plan_chain.invoke({"price": current_price, "threshold": PRICE_THRESHOLD}).strip()
    print(f"[PLAN] Decision: {decision}")
    return decision

# quick test
plan(105.0)


[PLAN] Decision: WAIT


'WAIT'

## 3. ACT
"Do it."

Here: if the decision was SEND_EMAIL, send the alert. (We just print it for the demo — swap in real `smtplib` code when you deploy this.)

In [14]:
def act(decision, current_price):
    """Carry out the decision from the Plan step."""
    if decision == "SEND_EMAIL":
        # --- in a real version, send an actual email here, e.g. via smtplib ---
        print(f"[ACT] 📧 Sending email: 'Alert! Price dropped to Rs {current_price}'")
        return "email_sent"
    else:
        print("[ACT] Nothing to do — price is fine.")
        return "no_action"

# quick test
act("SEND_EMAIL", 95.0)


[ACT] 📧 Sending email: 'Alert! Price dropped to Rs 95.0'


'email_sent'

## 4. OBSERVE
"See what came back."

Here: just confirm what happened as a result of the Act step.

In [5]:
def observe(action_result):
    """Record / check the result of the action we just took."""
    print(f"[OBSERVE] Result of action: {action_result}")
    return action_result

# quick test
observe("email_sent")


[OBSERVE] Result of action: email_sent


'email_sent'

## 5. REFLECT
"Check progress — keep going or stop."

Here: if the email was sent, the goal is met -> STOP. Otherwise -> loop back to Perceive.

In [6]:
def reflect(action_result):
    """Decide: is the goal met (stop), or do we need to loop again (continue)?"""
    if action_result == "email_sent":
        print("[REFLECT] Goal met (alert sent). -> STOP")
        return "stop"
    else:
        print("[REFLECT] Goal not met yet. -> LOOP BACK to Perceive")
        return "continue"

# quick test
reflect("email_sent")


[REFLECT] Goal met (alert sent). -> STOP


'stop'

## The loop
Now put all 5 steps together, in order, exactly as in the diagram:

**Perceive -> Plan -> Act -> Observe -> Reflect -> (loop back OR stop)**

In [17]:
def run_agent_loop(max_iterations=5, wait_seconds=2):
    for i in range(1, max_iterations + 1):
        print(f"\\n----- Loop iteration {i} -----")

        current_price = perceive()          # 1. PERCEIVE
        decision      = plan(current_price) # 2. PLAN
        action_result = act(decision, current_price)  # 3. ACT
        observe(action_result)              # 4. OBSERVE
        status = reflect(action_result)     # 5. REFLECT

        if status == "stop":
            print("\\nAgent loop finished.")
            break

        time.sleep(wait_seconds)  # small pause before looping back to Perceive
    else:
        print("\\nReached max iterations without meeting the goal.")

run_agent_loop()


\n----- Loop iteration 1 -----
[PERCEIVE] Current price: Rs 118.53
[PLAN] Decision: WAIT
[ACT] Nothing to do — price is fine.
[OBSERVE] Result of action: no_action
[REFLECT] Goal not met yet. -> LOOP BACK to Perceive
\n----- Loop iteration 2 -----
[PERCEIVE] Current price: Rs 86.62
[PLAN] Decision: SEND_EMAIL
[ACT] 📧 Sending email: 'Alert! Price dropped to Rs 86.62'
[OBSERVE] Result of action: email_sent
[REFLECT] Goal met (alert sent). -> STOP
\nAgent loop finished.
